# Database Exploration

This notebook provides a starting point to connect to your local PostgreSQL database and run queries.

In [9]:
import pandas as pd
from IPython.display import display, HTML
import warnings

In [10]:
warnings.filterwarnings('ignore', category=UserWarning)

In [1]:
import psycopg2
import psycopg2.extras

# Database connection parameters (from your .env file)
db_params = {
    "dbname": "market",
    "user": "market_user",
    "password": "market_pass",
    "host": "localhost",  # Because you are running the notebook on your host machine to connect to the docker container
    "port": "5432"
}

# Connect to the database
try:
    conn = psycopg2.connect(**db_params)
    # Return rows as dictionaries so they are easier to read like JSON objects
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    print("Successfully connected to the database!")
except Exception as e:
    print(f"Error connecting to the database: {e}")

Successfully connected to the database!


### Example Query: List Tables
Let's check what tables are in our `public` schema.

In [15]:
try:
    conn = psycopg2.connect(**db_params)
    # Return rows as dictionaries so they are easier to read like JSON objects
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    query = """
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public';
    """

    cursor.execute(query)
    tables = cursor.fetchall()

    print(f"Found {len(tables)} tables:")
    for row in tables:
        print(row)

finally:
    if conn:
        conn.close()

Found 2 tables:
RealDictRow({'table_name': 'raw_prices'})
RealDictRow({'table_name': 'clean_prices'})


In [ ]:
if conn:
    conn.close()

### Seeing the Tables and First Five of Each Table

In [12]:

# Connect to the database using credentials from your .env file
conn = psycopg2.connect(
    host="127.0.0.1",
    port=5432,
    dbname="market",
    user="market_user",
    password="market_pass"
)
try:
    # Query to fetch all table names in the 'public' schema into a DataFrame directly
    tables_df = pd.read_sql("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public'
    """, conn)
    
    if tables_df.empty:
        print("No tables found in the database.")
        
    for table_name in tables_df['table_name']:
        # Print a header for each table
        display(HTML(f"<h3>Table: <code>{table_name}</code></h3>"))
        
        # Fetch the first 5 rows and load directly into a Pandas DataFrame
        df = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 5;", conn)
        
        if not df.empty:
            display(df)
        else:
            print("Table is empty.")
            
finally:
    # Always close the connection when done
    if conn:
        conn.close()
    

,ticker,date,open,high,low,close,volume
0,AAPL,2025-01-02,190.0000,193.5000,189.5000,192.3000,55000000
1,AAPL,2025-01-03,192.3000,194.1000,191.2000,193.6000,42000000
2,MSFT,2025-01-02,375.0000,379.4000,373.1000,378.9000,21000000
3,SPY,2021-03-08,356.3266,361.8901,356.0466,359.0710,123149200
4,SPY,2021-03-09,361.4140,363.9717,359.6778,360.1818,113633600


,ticker,date,open,high,low,close,volume
0,AAPL,2021-04-24,130.8271,131.6063,128.7233,128.7233,78657500
1,AAPL,2021-04-25,130.8271,131.6063,128.7233,128.7233,78657500
2,AAPL,2021-04-26,131.2167,131.5479,130.0869,131.3238,66905100
3,AAPL,2021-04-27,130.8953,131.8888,130.6226,131.4992,66015800
4,AAPL,2021-04-28,130.1063,131.5089,129.6193,130.8173,107760100


### Getting the Distinct Tickers from `clean_prices`

In [16]:
try:
    conn = psycopg2.connect(**db_params)

    query = """
        SELECT DISTINCT ticker
        FROM clean_prices
        ORDER BY ticker;
    """

    unique_tickers_df = pd.read_sql(query, conn)

    print(f"Found {len(unique_tickers_df)} unique tickers:")
    display(unique_tickers_df)

finally:
    if conn:
        conn.close()

Found 10 unique tickers:


,ticker
0,AAPL
1,AMAT
2,COST
3,DOW
4,EXPE
5,GE
6,META
7,MSFT
8,PLTR
9,SPY


In [17]:
try:
    conn = psycopg2.connect(**db_params)

    query = """
        SELECT DISTINCT ticker
        FROM raw_prices
        ORDER BY ticker;
    """

    unique_tickers_df = pd.read_sql(query, conn)

    print(f"Found {len(unique_tickers_df)} unique tickers:")
    display(unique_tickers_df)

finally:
    if conn:
        conn.close()

Found 11 unique tickers:


,ticker
0,AAPL
1,AMAT
2,COST
3,DOW
4,EXPE
5,FLY
6,GE
7,META
8,MSFT
9,PLTR


### Cleanup
Always remember to close the connection when you are done.

In [ ]:
# cursor.close()
# conn.close()